**Eksperyment 1: Wpływ funkcji aktywacji na aproksymację funkcji w zadaniu regresji**
**Testowane funkcje:**
- Sigmoid
- ReLu
- Swish

**Każdy eksperyment będzie miał taką samą architektóre sieci tzn:**
- 1 hidden layer
- 16 perceptronów per layer
- aktywacja wyjściowa to f(x) = x
- funkcja straty to MSE
- optymalizatior to SGD
- learning rate = 0.01 oraz 0.001
- ilość epok = 500
- batch = 32


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

from mlp_model.activations.activation import Activation
from mlp_model.activations.relu import ReLU
from mlp_model.activations.swish import Swish

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from mlp_model.activations.sigmoid import Sigmoid
from mlp_model.initialize_weights.xavier import XavierInitializer
from mlp_model.layers.hidden import Hidden
from mlp_model.layers.output_layer import OutputLayer
from mlp_model.losses.mse_loss import MSELoss
from mlp_model.mlp import MLP
from mlp_model.optimizers.sgd import SGDOptimizer
from trainer.trainer import MLPTrainer
from sklearn.model_selection import train_test_split
from visualisation.regression_visualise import plot_learning_curve, plot_regression_fit, \
    plot_weights_evolution, print_cv_summary

data_kind = "activation"
data_count = 1000 # 100, 500, 1000, 10000

train_filepath = Path(project_root) / "data/regression" / f"data.{data_kind}.train.{data_count}.csv"
df = pd.read_csv(train_filepath)
x = df["x"].to_numpy().reshape(-1, 1)  # shape: (1000, 1)
y = df["y"].to_numpy().reshape(-1, 1)  # shape: (1000, 1)
learning_rates = [0.001]

C:\Users\arczi\PycharmProjects\NeuralNetworks\visualize_plot\helo_helo.png


In [2]:
epochs = 200
folds = 5
batch_size = 32
test_size = 0.2
random_state = 42
loss_function = MSELoss()
lr = 0.005

In [3]:
def _build_activation_mlp(activation: Activation) -> MLP:
    layers = [
        Hidden(1, 16, activation),
        OutputLayer(16, 1)
    ]
    mlp = MLP(layers=layers)
    mlp.initialize_weights(XavierInitializer())
    return mlp


def cross_validation(activation: Activation, filename: str | None = None):
    mlp = _build_activation_mlp(activation)
    sgd = SGDOptimizer(mlp, lr)
    trainer = MLPTrainer(mlp, sgd, loss_function, epochs, batch_size)
    summary = trainer.cross_validate(x, y, folds)
    print_cv_summary(summary, filename)


def visualization(activation: Activation, filename: str | None = None):
    mlp = _build_activation_mlp(activation)
    sgd = SGDOptimizer(mlp, lr)
    trainer = MLPTrainer(mlp, sgd, loss_function, epochs, batch_size)

    X_train, X_val, y_train, y_val = train_test_split(
        x,
        y,
        test_size=test_size,
        random_state=random_state,
        shuffle=True,
    )
    trainer.fit(X_train, y_train, X_val, y_val)

    plot_learning_curve(trainer.history, filename)
    plot_regression_fit(trainer.history, mlp, X_train, y_train, X_val, y_val, filename)
    for i in range(len(mlp.layers)):
        plot_weights_evolution(trainer.history, i, 16, filename)


def test_activation(activation: Activation, filename: str | None = None):
    cross_validation(activation, filename)
    visualization(activation, filename)

In [4]:
learning_rates = [0.005, 0.0005]
for l in learning_rates:
    lr = l
    test_activation(ReLU(), f"/regression/activation/ReLU-{lr}/relu")
    test_activation(Sigmoid(), f"/regression/activation/Sigmoid-{lr}/sigmoid")
    test_activation(Swish(), f"/regression/activation/Swish-{lr}/swish")

Fold 1: loss = 5.359914 | metrics {'r2': 0.9992981770366416, 'mae': 1.714801417355504, 'mse': 5.359913674461236}
Fold 2: loss = 10.459706 | metrics {'r2': 0.9986429392669526, 'mae': 2.8144762325297292, 'mse': 10.459705924586796}
Fold 3: loss = 29.993457 | metrics {'r2': 0.99597713997244, 'mae': 3.5042798268312207, 'mse': 29.993456924085127}
Fold 4: loss = 26.956198 | metrics {'r2': 0.9967743992552697, 'mae': 4.595693123069865, 'mse': 26.956198402398215}
Fold 5: loss = 1.361716 | metrics {'r2': 0.9998371560432708, 'mae': 0.8769365987210551, 'mse': 1.361716311152823}
---------------------------------------------------------------
Cross validation with 5 folds:
Mean loss: 14.826198 (std: 11.551205)
Mean metrics:
  R2: 0.998106 (std: 0.001484)
  MAE: 2.701237 (std: 1.307501)
  MSE: 14.826198 (std: 11.551205)
---------------------------------------------------------------
Fold 1: loss = 0.046698 | metrics {'r2': 0.9999938853884467, 'mae': 0.16018816603829308, 'mse': 0.04669808739483356}
Fol